In [8]:
# ============================================================================
# 1. 설정 및 라이브러리
# ============================================================================

import os
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone
from openai import OpenAI
import re

# .env 파일 로드
env_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path.cwd() / '.env'
load_dotenv(env_path)

# 환경 변수에서 API 키 로드
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
INDEX_NAME = "acquiror"  # target 인덱스 사용

# 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
index = pc.Index(INDEX_NAME)

print(f"[OK] Pinecone '{INDEX_NAME}' 인덱스 연결 완료")
print(index.describe_index_stats())

[OK] Pinecone 'acquiror' 인덱스 연결 완료
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '194',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 09 Feb 2026 07:29:06 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '56',
                                    'x-pinecone-request-latency-ms': '55',
                                    'x-pinecone-response-duration-ms': '57'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 1128023}},
 'storageFullness': 0.0,
 'total_vector_count': 1128023,
 'vector_type': 'dense'}


In [9]:
# ============================================================================
# 2. 유틸리티 함수
# ============================================================================

def get_embedding(text: str, model: str = "text-embedding-3-large") -> list:
    """OpenAI 임베딩 생성"""
    text = text.replace("\n", " ").strip()
    if not text:
        return None
    
    response = openai_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding


def parse_ipc_codes(ipc_string: str) -> list:
    """
    IPC 코드 문자열을 파싱하여 상위 클래스 리스트 반환
    예: "A61K 31/501|A61P 9/04" -> ["A61K", "A61P"]
    """
    if not ipc_string:
        return []
    
    # IPC 코드 패턴: 대문자+숫자+대문자 (예: A61K, B32B, E06B)
    pattern = r'([A-Z]\d{2}[A-Z])'
    matches = re.findall(pattern, ipc_string)
    
    # 중복 제거하고 리스트로 반환
    return list(set(matches))


def create_search_text(abstract: str, invention_name: str) -> str:
    """검색용 텍스트 생성 (초록 + 특허명 결합)"""
    parts = []
    if invention_name:
        parts.append(f"특허명: {invention_name}")
    if abstract:
        parts.append(f"초록: {abstract}")
    return " ".join(parts)


print("[OK] 유틸리티 함수 정의 완료")

[OK] 유틸리티 함수 정의 완료


In [12]:
# ============================================================================
# 3. 유사 특허 검색 함수
# ============================================================================

def find_similar_patents(
    ipc_code: str,
    abstract: str,
    invention_name: str,
    top_k: int = 5,
    use_ipc_filter: bool = True
) -> list:
    """
    IPC 코드로 1차 필터링 후, 초록+특허명 유사도로 검색
    
    Args:
        ipc_code: IPC 코드 문자열 (예: "A61K 31/501|A61P 9/04")
        abstract: 초록
        invention_name: 특허명
        top_k: 반환할 결과 수 (기본 5)
        use_ipc_filter: IPC 필터링 사용 여부 (기본 True)
    
    Returns:
        유사한 특허 리스트
    """
    
    # 1. 검색용 텍스트 생성 및 임베딩
    search_text = create_search_text(abstract, invention_name)
    query_embedding = get_embedding(search_text)
    
    if not query_embedding:
        print("[ERROR] 임베딩 생성 실패")
        return []
    
    # 2. IPC 코드 파싱 (상위 클래스 추출)
    ipc_classes = parse_ipc_codes(ipc_code)
    print(f"[INFO] 입력 IPC 클래스: {ipc_classes}")
    
    # 3. Pinecone 검색 (IPC 필터링 적용)
    filter_condition = None
    
    if use_ipc_filter and ipc_classes:
        # IPC 코드가 포함된 항목만 필터링
        # $or 조건으로 여러 IPC 클래스 중 하나라도 포함되면 매칭
        filter_condition = {
            "$or": [
                {"ipc_code": {"$contains": ipc_class}} 
                for ipc_class in ipc_classes
            ]
        }
        print(f"[FILTER] IPC 필터 적용: {ipc_classes}")
    
    # 4. 유사도 검색 실행
    try:
        results = index.query(
            vector=query_embedding,
            top_k=top_k * 2 if use_ipc_filter else top_k,  # 필터링 시 더 많이 검색
            include_metadata=True,
            filter=filter_condition
        )
    except Exception as e:
        # 필터 실패 시 필터 없이 재시도
        print(f"[WARN] IPC 필터링 실패, 필터 없이 검색: {e}")
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
    
    # 5. 결과 정리
    similar_patents = []
    for match in results.matches[:top_k]:
        patent = {
            "score": match.score,
            "id": match.id,
            "target_short_name": match.metadata.get("target_short_name", "N/A"),
            "target_id": match.metadata.get("target_id", "N/A"),
            "invention_name": match.metadata.get("invention_name", "N/A"),
            "abstract": match.metadata.get("abstract", "N/A"),
            "ipc_code": match.metadata.get("ipc_code", "N/A"),
            "application_number": match.metadata.get("application_number", "N/A"),
            "application_date": match.metadata.get("application_date", "N/A"),
        }
        similar_patents.append(patent)
    
    return similar_patents


def print_results(results: list):
    """검색 결과를 보기 좋게 출력"""
    if not results:
        print("[ERROR] 검색 결과 없음")
        return
    
    print(f"\n{'='*80}")
    print(f"[RESULT] Acquiror 유사 특허 검색 결과 (상위 {len(results)}개)")
    print(f"{'='*80}\n")
    
    for i, patent in enumerate(results, 1):
        print(f"[{i}] 유사도: {patent['score']:.4f}")
        print(f"    Target: {patent['target_short_name']} (ID: {patent['target_id']})")
        print(f"    특허명: {patent['invention_name'][:80]}...")
        print(f"    IPC: {patent['ipc_code'][:50]}...")
        print(f"    초록: {patent['abstract'][:150]}...")
        print(f"    출원번호: {patent['application_number']}")
        print(f"    출원일: {patent['application_date']}")
        print(f"    {'-'*70}")
    
    print()


print("[OK] 검색 함수 정의 완료")

[OK] 검색 함수 정의 완료


In [13]:
# ============================================================================
# 4. 사용 예시
# ============================================================================

# 검색할 특허 정보 입력
input_patent = {
    "ipc_code": "A61K 39/05",
    "invention_name": "PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT",  # 특허명
    "abstract": "The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin 1 of P. acnes (DsA1 polypeptide), Dermatan sulfate-binding adhesin 2 of P. acnes (DsA2 polypeptide), and putative iron-transport protein (PITP) polypeptide of P. acnes, and/or a fragment and/or derivative of DsA1 and/or DsA2 and/or PITP, wherein the DsA1 polypeptide and the DsA2 polypeptide comprise from N- to C-terminus an N-terminal swapping region (“NSR”), a first conserved sub-domain (“CSD1”), a first swapping region (“SR1”), a second conserved sub-domain (“CSD2”), a second swapping region (“SR2”), a third conserved sub-domain (“CSD3”), a Pro-Thr repeat containing region (“PT repeat region”), and a C-terminal region (“CTR”), and wherein the PITP polypeptide comprises from N- to C-terminus an extended neocarzinostatin family domain (“ENFD”), a first swapping region (“SR1”), a heme-binding domain (“HbD”), a second swapping region (“SR2”) including the C-terminal LPXTG motif, and a hydrophobic C-terminal region (“HLAR”)."
}

print("[INPUT] 입력 특허 정보:")
print(f"   IPC: {input_patent['ipc_code']}")
print(f"   특허명: {input_patent['invention_name']}")
print(f"   초록: {input_patent['abstract'][:100]}...")
print()

# 유사 특허 검색 실행
results = find_similar_patents(
    ipc_code=input_patent["ipc_code"],
    abstract=input_patent["abstract"],
    invention_name=input_patent["invention_name"],
    top_k=5,
    use_ipc_filter=True  # IPC 필터링 사용
)

# 결과 출력
print_results(results)

[INPUT] 입력 특허 정보:
   IPC: A61K 39/05
   특허명: PROPIONIBACTERIUM ACNES PROPHYLACTIC AND THERAPEUTIC IMMUNE TREATMENT
   초록: The present invention discloses a vaccine comprising one or more of Dermatan sulfate-binding adhesin...

[INFO] 입력 IPC 클래스: ['A61K']
[FILTER] IPC 필터 적용: ['A61K']
[WARN] IPC 필터링 실패, 필터 없이 검색: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 09 Feb 2026 07:34:35 GMT', 'Content-Type': 'application/json', 'Content-Length': '69', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '596', 'x-envoy-upstream-service-time': '71', 'x-pinecone-response-duration-ms': '598', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"$contains is not a valid operator","details":[]}


[RESULT] Acquiror 유사 특허 검색 결과 (상위 5개)

[1] 유사도: 0.6457
    Target: N/A (ID: N/A)
    특허명: NOVEL SKIN CARE COMPOSITION...
    IPC: A61K35/74|A61K47/22|A61K8/49|A61K47/14|A61K8/37|A6...
    초록:  The present invention generally relates to the field of skin care. 